Lanette Tyler   
ST554   
Spring 2026   
Project 3

# Fitting the Model

## Preliminary Steps: Import Modules and Functions, Create Spark Session

In [85]:
#import module 
import pandas as pd
import numpy as np

In [40]:
#import functions
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, PCA, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator, CrossValidatorModel
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.types import StructType
from pyspark.sql.functions import col

In [8]:
#create spark session
spark = SparkSession.builder.getOrCreate()

## Read in Data

In [9]:
#read in the project data to a standard pandas data frame and view it
power_ml_data = pd.read_csv("https://www4.stat.ncsu.edu/online/datasets/power_ml_data.csv")
power_ml_data.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


In [10]:
power_ml_data.shape #check data set size

(47174, 10)

There are 47,174 observations in the data set.

In [11]:
#convert to a spark SQL data frame
power_ml_data = spark.createDataFrame(power_ml_data)
power_ml_data.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

## Transform Data

### Hour Column to Double Type, Response Variable Power_Zone_3 Renamed 'label'

In [12]:
#check data type of Hour column
power_ml_data

DataFrame[Temperature: double, Humidity: double, Wind_Speed: double, General_Diffuse_Flows: double, Diffuse_Flows: double, Power_Zone_1: double, Power_Zone_2: double, Power_Zone_3: double, Month: bigint, Hour: bigint]

The Hour column is read in as data type "bigint." It needs to be changed to "double" type.

In [13]:
#transform data type of Hour column to "double" type
SQLtrans = SQLTransformer(
                statement = """
                    SELECT Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, Diffuse_Flows,
                           Power_Zone_1, Power_Zone_2, Power_Zone_3 AS label, Month, CAST(HOUR AS DOUBLE) AS Hour
                    FROM __THIS__
                    """)

In [14]:
#view transformation results and object
SQLtrans.transform(power_ml_data).show(10)
SQLtrans.transform(power_ml_data)

+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538|20240.96386|    1| 0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599|20131.08434|    1| 0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693|19668.43373|    1| 0.0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422|18899.27711|    1| 0.0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043|18442.40964|    1| 0.0|
|      5.853|    76.9|     0.081|               

DataFrame[Temperature: double, Humidity: double, Wind_Speed: double, General_Diffuse_Flows: double, Diffuse_Flows: double, Power_Zone_1: double, Power_Zone_2: double, label: double, Month: bigint, Hour: double]

The column Hour is now cast to double data type.

### Hour Column to Binary

In [15]:
#make Hour column binary, Hour < 6.5
binaryTrans = Binarizer(threshold = 6.5, inputCols = ["Hour"], 
                        outputCols = ["Daytime"])
binaryTrans.transform(
    SQLtrans.transform(power_ml_data)).show(10)

+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Month|Hour|Daytime|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538|20240.96386|    1| 0.0|    0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599|20131.08434|    1| 0.0|    0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693|19668.43373|    1| 0.0|    0.0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422|18899.27711|    1| 0.0|    0.0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043|18442.40964|    

### Month Column One-Hot Encoded

In [16]:
#define One Hot Encoder trnasformation/model, and view object
OHEtrans = OneHotEncoder(inputCol = "Month", outputCol = "MonthVec")
OHEtrans

OneHotEncoder_95e1bcf82642

In [17]:
OHEmodel = OHEtrans.fit(
                binaryTrans.transform(
                    SQLtrans.transform(power_ml_data)))
OHEmodel

OneHotEncoderModel: uid=OneHotEncoder_95e1bcf82642, dropLast=true, handleInvalid=error

In [18]:
#fit the model, and view object
OHEmodel = OHEtrans.fit(power_ml_data)
OHEmodel

OneHotEncoderModel: uid=OneHotEncoder_95e1bcf82642, dropLast=true, handleInvalid=error

In [19]:
#tansform data with model fit plus previous transformations, and view the object
OHEmodel.transform(
            binaryTrans.transform(
                SQLtrans.transform(power_ml_data)))

DataFrame[Temperature: double, Humidity: double, Wind_Speed: double, General_Diffuse_Flows: double, Diffuse_Flows: double, Power_Zone_1: double, Power_Zone_2: double, label: double, Month: bigint, Hour: double, Daytime: double, MonthVec: vector]

In [20]:
#apply transformations to the data set and view results
OHEmodel.transform(
            binaryTrans.transform(
                SQLtrans.transform(power_ml_data))) \
    .show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Month|Hour|Daytime|      MonthVec|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538|20240.96386|    1| 0.0|    0.0|(12,[1],[1.0])|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599|20131.08434|    1| 0.0|    0.0|(12,[1],[1.0])|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693|19668.43373|    1| 0.0|    0.0|(12,[1],[1.0])|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422|18899.27711|    1| 0.0|    0.0|(12,[1],[1.0])|
|     

### PCA Fit

In [21]:
#create features column for principal components analysis
PCAassembler = VectorAssembler(inputCols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"], 
                            outputCol = "PCAfeatures", handleInvalid = "keep")

In [22]:
PCAassembler.transform(
            OHEmodel.transform(
                    binaryTrans.transform(
                        SQLtrans.transform(power_ml_data)))) \
    .show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+--------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Month|Hour|Daytime|      MonthVec|         PCAfeatures|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+--------------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538|20240.96386|    1| 0.0|    0.0|(12,[1],[1.0])|[6.559,73.8,0.083...|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599|20131.08434|    1| 0.0|    0.0|(12,[1],[1.0])|[6.414,74.5,0.083...|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693|19668.43373|    1| 0.0|    0.0|(12,[1],[1.0])|[6.313,74.5,0.08,...|
|      6.121|    75.0|

In [23]:
PCAtrans = PCA(k = 2, inputCol = "PCAfeatures", outputCol = "PCAresult")
PCAtrans

PCA_d674fa1a1203

In [24]:
PCAmodel = PCAtrans.fit(
                PCAassembler.transform(
                    OHEmodel.transform(
                        binaryTrans.transform(
                            SQLtrans.transform(power_ml_data)))))
PCAmodel

PCAModel: uid=PCA_d674fa1a1203, k=2

In [25]:
PCAmodel.transform(
                PCAassembler.transform(
                    OHEmodel.transform(
                        binaryTrans.transform(
                            SQLtrans.transform(power_ml_data))))) \
    .show(10)

+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+--------------------+--------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Month|Hour|Daytime|      MonthVec|         PCAfeatures|           PCAresult|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+--------------------+--------------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538|20240.96386|    1| 0.0|    0.0|(12,[1],[1.0])|[6.559,73.8,0.083...|[1.79440486365695...|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599|20131.08434|    1| 0.0|    0.0|(12,[1],[1.0])|[6.414,74.5,0.083...|[1.80604083009823...|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.1012

In [26]:
#view principal components matrix
PCAmodel.pc

DenseMatrix(5, 2, [-0.0095, 0.0263, -0.001, -0.9553, -0.2943, 0.0075, -0.0094, 0.0025, 0.2941, -0.9557], 0)

In [27]:
PCAmodel.explainedVariance

DenseVector([0.8839, 0.1136])

### Final Features Column for Transformed Data

In [28]:
#create features column for final assembly
assembler = VectorAssembler(inputCols = ["PCAresult", "Daytime", "Power_Zone_1", "Power_Zone_2", "MonthVec"], 
                            outputCol = "features", handleInvalid = "keep")
assembler

VectorAssembler_4e54f62d275c

In [29]:
#view results
assembler.transform(
        PCAmodel.transform(
                PCAassembler.transform(
                    OHEmodel.transform(
                        binaryTrans.transform(
                            SQLtrans.transform(power_ml_data)))))) \
    .select("features").show(10, truncate = False)

+-------------------------------------------------------------------------------------+
|features                                                                             |
+-------------------------------------------------------------------------------------+
|(17,[0,1,3,4,6],[1.7944048636569538,-0.741274644740427,34055.6962,16128.87538,1.0])  |
|(17,[0,1,3,4,6],[1.806040830098231,-0.7108534239558292,29814.68354,19375.07599,1.0]) |
|(17,[0,1,3,4,6],[1.8102297630563897,-0.728311319115875,29128.10127,19006.68693,1.0]) |
|(17,[0,1,3,4,6],[1.7986676517408837,-0.7220913032199756,28228.86076,18361.09422,1.0])|
|(17,[0,1,3,4,6],[1.8632872016379705,-0.7323046647776371,27335.6962,17872.34043,1.0]) |
|(17,[0,1,3,4,6],[1.8782067450046098,-0.7628199906979403,26624.81013,17416.41337,1.0])|
|(17,[0,1,3,4,6],[1.9152929871795548,-0.7636928919816222,25998.98734,16993.31307,1.0])|
|(17,[0,1,3,4,6],[1.9240054080702906,-0.7645388021423595,25446.07595,16661.39818,1.0])|
|(17,[0,1,3,4,6],[1.895019303530

In [30]:
#view bottom results
assembler.transform(
        PCAmodel.transform(
                PCAassembler.transform(
                    OHEmodel.transform(
                        binaryTrans.transform(
                            SQLtrans.transform(power_ml_data)))))) \
    .select("features").tail(10)

[Row(features=SparseVector(17, {0: 1.683, 1: -0.6972, 2: 1.0, 3: 34737.6426, 4: 29132.8628})),
 Row(features=SparseVector(17, {0: 1.706, 1: -0.6905, 2: 1.0, 3: 33776.4259, 4: 28230.7456})),
 Row(features=SparseVector(17, {0: 1.7094, 1: -0.6881, 2: 1.0, 3: 33387.0722, 4: 27814.6671})),
 Row(features=SparseVector(17, {0: 1.7267, 1: -0.7132, 2: 1.0, 3: 32815.2091, 4: 27564.2835})),
 Row(features=SparseVector(17, {0: 1.7554, 1: -0.698, 2: 1.0, 3: 32158.1749, 4: 27273.3968})),
 Row(features=SparseVector(17, {0: 1.7706, 1: -0.706, 2: 1.0, 3: 31160.4563, 4: 26857.3182})),
 Row(features=SparseVector(17, {0: 1.7668, 1: -0.7022, 2: 1.0, 3: 30430.4182, 4: 26124.5781})),
 Row(features=SparseVector(17, {0: 1.7466, 1: -0.6766, 2: 1.0, 3: 29590.8745, 4: 25277.6925})),
 Row(features=SparseVector(17, {0: 1.766, 1: -0.6992, 2: 1.0, 3: 28958.1749, 4: 24692.2369})),
 Row(features=SparseVector(17, {0: 1.7939, 1: -0.7331, 2: 1.0, 3: 28349.8099, 4: 24055.2317}))]

### Pipeline

In [31]:
#define pipeline for transforming data, view pipeline object
pipeline = Pipeline(stages = [SQLtrans, binaryTrans, OHEtrans, PCAassembler, PCAtrans, assembler])
pipeline

Pipeline_c4802ebf0dfd

In [32]:
#make pipeline model, view pipeline model object
pModel = pipeline.fit(power_ml_data)
pModel

PipelineModel_f4bad06c8029

In [33]:
#transform data using pipeline model, view transformed data object
trData = pModel.transform(power_ml_data)
trData

DataFrame[Temperature: double, Humidity: double, Wind_Speed: double, General_Diffuse_Flows: double, Diffuse_Flows: double, Power_Zone_1: double, Power_Zone_2: double, label: double, Month: bigint, Hour: double, Daytime: double, MonthVec: vector, PCAfeatures: vector, PCAresult: vector, features: vector]

In [34]:
#view transformed data
trData.show(10)

+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+--------------------+--------------------+--------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Month|Hour|Daytime|      MonthVec|         PCAfeatures|           PCAresult|            features|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+--------------------+--------------------+--------------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538|20240.96386|    1| 0.0|    0.0|(12,[1],[1.0])|[6.559,73.8,0.083...|[1.79440486365695...|(17,[0,1,3,4,6],[...|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599|20131.08434|    1| 0.0|    0.0|(12,[1],[1.0])|[6.414,74.5,0.083...|[1.80604083009823.

In [35]:
#view untruncated features column of transformed data
pModel.transform(power_ml_data).select("features").show(10, truncate = False)

+-------------------------------------------------------------------------------------+
|features                                                                             |
+-------------------------------------------------------------------------------------+
|(17,[0,1,3,4,6],[1.7944048636569538,-0.7412746447404435,34055.6962,16128.87538,1.0]) |
|(17,[0,1,3,4,6],[1.806040830098231,-0.7108534239558459,29814.68354,19375.07599,1.0]) |
|(17,[0,1,3,4,6],[1.81022976305639,-0.7283113191158916,29128.10127,19006.68693,1.0])  |
|(17,[0,1,3,4,6],[1.798667651740884,-0.7220913032199924,28228.86076,18361.09422,1.0]) |
|(17,[0,1,3,4,6],[1.8632872016379707,-0.7323046647776539,27335.6962,17872.34043,1.0]) |
|(17,[0,1,3,4,6],[1.8782067450046103,-0.7628199906979573,26624.81013,17416.41337,1.0])|
|(17,[0,1,3,4,6],[1.9152929871795552,-0.7636928919816394,25998.98734,16993.31307,1.0])|
|(17,[0,1,3,4,6],[1.9240054080702913,-0.7645388021423769,25446.07595,16661.39818,1.0])|
|(17,[0,1,3,4,6],[1.895019303530

The transformed data produced by the pipeline looks the same as the transformed data generated above manually using the piepeline steps.

## Fit Elastic Net Linear Regression Model to Transformed Data

In [36]:
#define linear regression model
lr = LinearRegression(solver = "l-bfgs") #for large data sets

In [37]:
#set parameter grid for regularization and elastic net parameters
grid = ParamGridBuilder() \
  .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
  .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
  .build()

In [38]:
cv = CrossValidator(estimator = lr,
                    estimatorParamMaps = grid,
                    evaluator = RegressionEvaluator(metricName = "rmse"),
                    numFolds = 5,
                    parallelism = 128)
cv

CrossValidator_6461fa087987

In [48]:
#use the cross validator object to fit the model to the transformed data
cvModel = cv.fit(trData)

26/04/27 16:37:05 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [42]:
#view crossvalidator model object
cvModel

CrossValidatorModel_f8530754c10a

In [52]:
#save the model in current working directory
cvModel.write().overwrite().save("p3Model")

In [41]:
#load the saved model from current working directory
cvModel = CrossValidatorModel.load("p3Model")

In [43]:
#view CV error and corresponding tuning parameter of models fit, first few values
my_list = []
for i in range(len(grid)):
    my_list.append([cvModel.avgMetrics[i], grid[i].values()])
print("RMSE, [regParam, elasticNetParam]")
my_list[:10]

RMSE, [regParam, elasticNetParam]


[[2147.783898438477, dict_values([0.0, 0.0])],
 [2147.7838984384775, dict_values([0.0, 0.05])],
 [2147.7838984384775, dict_values([0.0, 0.1])],
 [2147.7838984384775, dict_values([0.0, 0.25])],
 [2147.7838984384775, dict_values([0.0, 0.5])],
 [2147.7838984384775, dict_values([0.0, 0.75])],
 [2147.783898438477, dict_values([0.0, 0.9])],
 [2147.7838984384775, dict_values([0.0, 0.95])],
 [2147.7838984384775, dict_values([0.0, 0.98])],
 [2147.7838984384775, dict_values([0.0, 0.99])]]

In [48]:
#extract tuning parameters for best fit, which are stored in cvModel
#find model fit with lowest error
comp = my_list[0][0]
for i in my_list:
    if i[0] < comp:
        comp = i[0]
        current_min = i
print("Average RMSE, Regularization Parameter, and Elastic Net Parameter for Best Fit CV Model:")
print([current_min])

Average RMSE, Regularization Parameter, and Elastic Net Parameter for Best Fit CV Model:
[[2147.783556202275, dict_values([0.05, 0.5])]]


In [45]:
#Use model to transform create and evaluate predictions on entire data set (training RSME)
preds = cvModel.transform(trData)
print(preds) #view data frame object
preds.show(5) #view first few rows

DataFrame[Temperature: double, Humidity: double, Wind_Speed: double, General_Diffuse_Flows: double, Diffuse_Flows: double, Power_Zone_1: double, Power_Zone_2: double, label: double, Month: bigint, Hour: double, Daytime: double, MonthVec: vector, PCAfeatures: vector, PCAresult: vector, features: vector, prediction: double]
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+--------------------+--------------------+--------------------+------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Month|Hour|Daytime|      MonthVec|         PCAfeatures|           PCAresult|            features|        prediction|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+--------------------+--------------------+--------------------+------------------+
|   

In [46]:
#evaluate predictions with training RMSE
RegressionEvaluator().evaluate(preds)

2147.09734598245

The trianing RMSE is almost the same as the CV RMSE's above, but just slightly better.

In [49]:
#residuals data frame with predictions, label, and residuals(label - prediction)
predsWithResDF = preds.select("label", "prediction") \
                .withColumn("residuals", col("label") - col("prediction"))
predsWithResDF.show(10)

+-----------+------------------+------------------+
|      label|        prediction|         residuals|
+-----------+------------------+------------------+
|20240.96386| 20880.29948851184|-639.3356285118389|
|20131.08434|  18660.0090763708|1471.0752636292018|
|19668.43373| 18204.51422991327|1463.9195000867294|
|18899.27711|17590.430361135004|1308.8467488649949|
|18442.40964|16997.060832471776|1445.3488075282257|
|18130.12048|16517.445986659906| 1612.674493340095|
|17945.06024|  16093.0118402874|1852.0483997125993|
|17459.27711| 15722.45502627943|  1736.82208372057|
|17025.54217| 15270.80540469168|1754.7367653083202|
|16794.21687|14938.101832724918|1856.1150372750817|
+-----------+------------------+------------------+
only showing top 10 rows


# Streaming Part

## Reading a Stream

A folder called csv_files is set up in the same directory as the .py file for creating the streaming data.

In [ ]:
DataFrame[Temperature: double, Humidity: double, Wind_Speed: double, General_Diffuse_Flows: double, Diffuse_Flows: double, Power_Zone_1: double, Power_Zone_2: double, Power_Zone_3: double, Month: bigint, Hour: bigint]

In [69]:
#set up schema for data stream
my_schema = StructType() \
                .add("Temperature", "double") \
                .add("Humidity", "double") \
                .add("Wind_Speed", "double") \
                .add("General_Diffuse_Flows", "double") \
                .add("Diffuse_Flows", "double") \
                .add("Power_Zone_1", "double") \
                .add("Power_Zone_2", "double") \
                .add("Power_Zone_3", "double") \
                .add("Month", "integer") \
                .add("Hour", "integer")

In [73]:
#listen for streaming data coming into csv_files
input = spark.readStream.schema(my_schema).option("header", "true").csv("csv_files")
print(input)

DataFrame[Temperature: double, Humidity: double, Wind_Speed: double, General_Diffuse_Flows: double, Diffuse_Flows: double, Power_Zone_1: double, Power_Zone_2: double, Power_Zone_3: double, Month: int, Hour: int]


## Transform/Aggregation Step

In [103]:
#use pipeline transformer to transform incoming data and model transformer to obtain predictions on the transformed data
#this is the first transformation/aggregation step for the streaming part

trInput1 = pModel.transform(input) #pipeline transformer
str_preds = cvModel.transform(trInput1) #cv model transformer
strPredsWithResDF = str_preds.select("label", "prediction") \
                        .withColumn("residuals", col("label") - col("prediction"))
strPredsWithResDF

DataFrame[label: double, prediction: double, residuals: double]

In [79]:
#rename response variable of incoming data to label
#second transformation
trInput2 = input.withColumnRenamed("Power_Zone_3", "label")
trInput2

DataFrame[Temperature: double, Humidity: double, Wind_Speed: double, General_Diffuse_Flows: double, Diffuse_Flows: double, Power_Zone_1: double, Power_Zone_2: double, label: double, Month: int, Hour: int]

In [104]:
#join the two input streams together on "label"column
joined_input = strPredsWithResDF.join(trInput2, "label", "inner")
joined_input

DataFrame[label: double, prediction: double, residuals: double, Temperature: double, Humidity: double, Wind_Speed: double, General_Diffuse_Flows: double, Diffuse_Flows: double, Power_Zone_1: double, Power_Zone_2: double, Month: int, Hour: int]

## Writing Step

In [108]:
my_query = joined_input \
                .writeStream.outputMode("append") \
                .format("console") \
                .start()

26/04/27 23:01:32 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-84e01e12-3d59-4187-9072-e9ba167cd79a. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/27 23:01:32 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|         residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16036.62651|14443.951218143797|1592.6752918562033|      11.06|    75.7|     4.915|                0.077|         0.13| 23775.18987| 14184.80243|    1|   1|
|8966.537283| 8890.901815274388|  75.6354677256113|      21.85|   46.22|     4.916|                43.06|         34.0| 24664.77876| 14456.13306|    9|   7|
|20686.57814| 24652.79709139546|-3966.218951395458|      22.14|    80.6|     4.922|                23.61|        19.43

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|19993.67781|22442.696016830192| -2449.018206830191|      18.99|    89.5|     0.101|                 45.2|        40.61|  47006.3895| 30006.22407|   10|  18|
|19089.45455|20492.513260766438|-1403.0587107664396|      14.08|    90.0|      0.07|                103.0|         93.4| 34777.00753| 18894.50102|    4|  11|
|12331.33253|13007.632582826627| -676.3000528266275|      10.44|    89.3|     0.083|                445.7|       

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|         residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|18214.83871|17984.687474095717|230.15123590428266|      10.66|    89.8|     0.076|                0.048|        0.148| 29792.68085| 18113.41463|    3|   0|
|19957.59036|15050.346822809915| 4907.243537190083|      12.44|    83.3|     0.083|                 0.04|        0.115| 26782.78481| 19232.82675|    1|  23|
| 33644.1841| 30686.08160665768|2958.1024933423178|      36.04|   27.41|     4.908|                837.0|         98.9

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residuals|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|12722.18845|13524.844383716332| -802.6559337163326|      21.68|    72.4|     4.921|                613.8|        289.6| 36645.95186| 23616.59751|   10|  13|
|18615.06073|16051.506107389374|  2563.554622610627|      28.23|   26.79|     0.081|                888.0|        46.86| 32923.27869| 17732.50774|    5|  13|
|12984.80243|14138.170115015055| -1153.367685015055|      23.77|   64.03|     0.068|                374.6|       

In [109]:
my_query.stop()

26/04/27 23:03:06 WARN DAGScheduler: Failed to cancel job group d1e5bc63-dfa7-46e4-8e4c-4bd6918742dd. Cannot find active jobs for it.
26/04/27 23:03:07 WARN DAGScheduler: Failed to cancel job group d1e5bc63-dfa7-46e4-8e4c-4bd6918742dd. Cannot find active jobs for it.


## Producing Data

In [102]:
#import modules
import time
import pandas as pd
import numpy as np

In [105]:
#testing code for .py file here
#create loop for outputting data to csv files to be read as streaming data
#randomly sample five rows of data at a time to write to csv files
#put in a pause between loops

pwr_str_data = pd.read_csv("power_streaming_data.csv")

for i in range(0, 20):
    temp = pwr_str_data.loc[np.random.randint(pwr_str_data.shape[0], size = 5)]
    temp.to_csv("csv_files/pwr_str_data" + str(i) + ".csv", index = False, header = True)
    time.sleep(15)

KeyboardInterrupt: 